In [1]:
import pandas as pd
import json
import numpy as np
import random

### APPLICATION

In [2]:
#APPLICATION
application = pd.read_csv("./data/APPLICATION.csv")

applications_list = []
for col in ["INDUSTRY_GROUP","MARKET","MARKET_SEGMENT","SUB_SEGMENT","APPLICATION"]:
    applications_list += application[col].dropna().unique().tolist()
    
del application

applications_list = [x.lower() for x in applications_list]
applications_list[0:5]

['industrial',
 'consumer goods',
 'automotive & transportation',
 'electrical & electronics',
 'medical & pharma']

In [3]:
SPT = pd.read_csv("./data/SPT.csv")

def remove_special_characters(text):
    return ''.join(e for e in text if e.isalnum())

def get_unique_values(df, column):
    return list(df[column][df[column].notna()].unique())

In [4]:
SPT.head()

,PRODUCT_LINE,POLYMER,PRODUCT_CD,PROPERTY_NAME,NON_STD_TEST_COND_DESC,VALUE_ASSMNT_SI,UNIT_OF_MEAS_SI,TEMP_C2
0,CELANYL®,PA66,CELANYL A3 D10 BK 9005/G,Melting temperature,20°C/min,260,°C,Thermal properties
1,CELANYL®,PA66,CELANYL A3 CF30 BK 9005,Tensile modulus,NaN,21500,MPa,Mechanical properties
2,CELANYL®,PA6,CELANYL B3 N NC 1102,Tensile stress at yield,50mm/min,78,MPa,Mechanical properties
3,GUR®,PE-UHMW,GUR X 240,Density,NaN,940,kg/m³,Physical properties
4,CELANYL®,PA66,CELANYL A3 D10 BK 9005/G,Density,NaN,1100,kg/m³,Physical properties


### BRAND

In [5]:
#BRAND

Brands = list(SPT[SPT['PRODUCT_LINE'].notna()]['PRODUCT_LINE'].apply(remove_special_characters).unique())
Brands+= [x.lower() for x in Brands]

### POLYMER

In [6]:
#POLYMER
Polymer = get_unique_values(SPT, 'POLYMER')
Polymer+= [x.lower() for x in Polymer]

### PROPERTY

In [7]:
TDS_MAPPING = pd.read_csv("./data/TDS_MAPPING.csv")
CTQ = pd.read_csv("./data/CTQ.csv")

In [8]:
property_list_new = ['Specific Resistivity','Volume resistivity','Surface resistivity','Dissipation factor','Tensile stress at break',
'Tensile strain at break','Total load','Izod impact notched','Emission of organic compounds','Glow wire flammability index',
'Glass fiber load','Melt volume rate','Poissons ratio','Water absorption','Burning rate','Comparative tracking index',
'Mineral filler load','Carbon fiber load','compressive stress']

In [9]:
for x in CTQ['SYNONYMS'].dropna().unique().tolist():
    property_list_new+=x.split(";") 
    
property_list_new+= CTQ['USER_ENTERS'].dropna().unique().tolist()

property_list_new = [x.strip().lower() for x in property_list_new]

In [10]:
td_property_list = get_unique_values(TDS_MAPPING, 'TDS_PROPERTY')

In [11]:
#PROPERTY
properties = pd.read_csv("./data/PROPERTY.csv")
property_list = get_unique_values(properties, 'PROPERTY_NAME')
property_list = [x.lower() for x in property_list]

additional_properties="Stiffness, Mechanical Strength, Mechanical Behavior, Mechanical Resistance, mechanical properties, Deformation, Deformation stability, Flexibility, Mechanical performance, Modulus, Strength, Tensile strength, Dense, Heavy, Impact, impact resistance, durability, tough, solidity, impact resistant, Dimensional, precise, dimensional precision, Impact, impact resistance, durability, tough, solidity, impact resistant, Thermal resistance, use temperature, Flexural performance, Rigid, Rigidity, Stiffness, Flow, Flowability, Flow rate, Toughness, Dimensional Stability, Toughness, Moisture absorption, Thermal expansion, Dissipation factor, Temperature performance, Electrical strength, Stiff, Stiffness at High Temp, Melt index"
additional_properties = additional_properties.lower()
additional_properties = additional_properties.split(", ")
property_list += additional_properties

property_list = list(set(property_list))

In [12]:
for prop in property_list:
    if ',' in prop:
        property_list_new.append(prop.split(",")[0])
    if '@ ' in prop:
        property_list_new.append(prop.replace('@',random.choice(['at','@'])))
    else:
        property_list_new.append(prop)
        
td_property_list = list(set([x.lower() for x in td_property_list]))
for prop in td_property_list:
    if ',' in prop:
        prop = prop.split(",")[0]
        if "(" in prop and ")" not in prop :
            prop = prop.split("(")[0]
    
        #print(prop)
    property_list_new.append(prop)

In [13]:
property_list_new = list(map(lambda x: x.replace("bar flow 320℃","bar flow at 320℃"), property_list_new))
property_list_new = list(map(lambda x: x.replace("heat deflection temperature 1.8mpa","heat deflection temperature at 1.8mpa"), property_list_new))
property_list_new = list(map(lambda x: x.replace("un notched charpy impact -30 edgewise","unnotched charpy impact -30 edgewise"), property_list_new))
property_list_new = list(map(lambda x: x.replace("compressive stress at 1 / 2 / 5 % nominal strain (10) -","compressive stress at 2% nominal strain"), property_list_new))
property_list_new = list(map(lambda x: x.replace("water absorption at saturation in water of 23 °c (73°f)","water absorption at 23 °c"), property_list_new))
property_list_new = list(map(lambda x: x.replace("limiting pv at 0.1 / 1 m/s cylindrical sleeve bearings","limiting pv at 0.1m/s cylindrical sleeve bearings"), property_list_new))
property_list_new = list(map(lambda x: x.replace("izod notched impact strength iso 180/a (23°c)","izod notched impact strength"), property_list_new))
property_list_new = list(map(lambda x: x.replace("surface resistivity 23°c 50% rh","surface resistivity"), property_list_new))
property_list_new = list(map(lambda x: x.replace("clte flow (23°c) -","clte flow (23°c)"), property_list_new))
property_list_new = list(map(lambda x: x.replace("molding shrinkage flow -","molding shrinkage flow"), property_list_new))
property_list_new = list(map(lambda x: x.replace("relative permittivity εr : at 1 mhz -","relative permittivity"), property_list_new))
property_list_new = list(map(lambda x: x.replace("notched izod impact 73°f(23°c)","notched izod impact"), property_list_new))
property_list_new = list(map(lambda x: x.replace("average mv(10 g/mol) 6","average mv"), property_list_new))
property_list_new = list(map(lambda x: x.replace("oxygen index 2","oxygen index"), property_list_new))
property_list_new = list(map(lambda x: x.replace("strain at break 80°c","strain at break"), property_list_new))
property_list_new = list(map(lambda x: x.replace("average molar mass (average molecular weight) (1)average molar mass (average molecular weight) (1)","average molar mass"), property_list_new))
property_list_new = list(map(lambda x: x.replace("drying time*","drying time"), property_list_new))
property_list_new = list(map(lambda x: x.replace("continuous allowable service temperature in air (20.000 hrs) (3)","continuous allowable service temperature in air"), property_list_new))
property_list_new = list(map(lambda x: x.replace("back pressure **","back pressure"), property_list_new))
property_list_new = list(map(lambda x: x.replace("heat distortion temperature 18.5kgf/cm2","heat distortion temperature"), property_list_new))
property_list_new = list(map(lambda x: x.replace("shear strength 23℃","shear strength at 23℃"), property_list_new))
property_list_new = list(map(lambda x: x.replace('relative volume loss during wear test sand-slurry" : tivar® 1000=100"','relative volume loss during wear test sand-slurry'), property_list_new))
property_list_new = list(map(lambda x: x.replace("compressive stress at 1 / 2 / 5 % nominal strain","compressive stress at nominal strain"), property_list_new))
property_list_new = list(map(lambda x: x.replace("burning behavior ul 94","burning behavior"), property_list_new))
property_list_new = list(map(lambda x: x.replace("tensile modulus of elasticity (10) -","tensile modulus of elasticity"), property_list_new))
property_list_new = list(map(lambda x: x.replace("deflection temperature at 0.46 mpa (66 psi)","deflection temperature at 0.46 mpa"), property_list_new))
property_list_new = list(map(lambda x: x.replace("dynamic coefficient of friction (-)","dynamic coefficient of friction"), property_list_new))
property_list_new = list(map(lambda x: x.replace("water vapor transmission rate at 23 â„ƒ /85%r.h.'","water vapor transmission rate"), property_list_new))
property_list_new = list(map(lambda x: x.replace("flexural modulus of elasticity -","flexural modulus of elasticity -"), property_list_new))
property_list_new = list(map(lambda x: x.replace("average spec. heat capacity 20-150 â„ƒ","average spec. heat capacity"), property_list_new))
property_list_new = list(map(lambda x: x.replace("melt volume-flow rate mvr at 190 °c and 10 kg","melt volume-flow rate mvr"), property_list_new))
property_list_new = list(map(lambda x: x.replace("heat deflection temperature 264psi","heat deflection temperature"), property_list_new))

property_list_new = list(map(lambda x: x.replace("insulation resistance 3 (90°c)","insulation resistance"), property_list_new))
property_list_new = list(map(lambda x: x.replace("min. service temperature (5)","min. service temperature"), property_list_new))
property_list_new = list(map(lambda x: x.replace("'unnotched izod impact 73°f (23°c)","'unnotched izod impact at 73°f"), property_list_new))

property_list_new = list(map(lambda x: x.replace("taber abrasion resistance 1000 cycles","taber abrasion resistance"), property_list_new))
property_list_new = list(map(lambda x: x.replace("flammability fmvss302","flammability"), property_list_new))
property_list_new = list(map(lambda x: x.replace("hydrolytic stability 2","hydrolytic stability"), property_list_new))
property_list_new = list(map(lambda x: x.replace("burning rate. thickness 1 mm","burning rate. thickness"), property_list_new))
property_list_new = list(map(lambda x: x.replace("strain at break 80c","strain at break"), property_list_new))
property_list_new = list(map(lambda x: x.replace("stress 10% deformation","stress at 10% deformation"), property_list_new))
property_list_new = list(map(lambda x: x.replace("insulation resistance 1 (90°c)","insulation resistance"), property_list_new))
property_list_new = list(map(lambda x: x.replace("min. service temperature (4)","min. service temperature"), property_list_new))

property_list_new = list(map(lambda x: x.replace("hdt 0.45mpa","hdt at 0.45mpa"), property_list_new))
property_list_new = list(map(lambda x: x.replace("flexural modulus of elasticity -","flexural modulus of elasticity"), property_list_new))
property_list_new = list(map(lambda x: x.replace("tensile modulus - chord 1","tensile modulus - chord"), property_list_new))
property_list_new = list(map(lambda x: x.replace("elongation at break 23℃","elongation at break"), property_list_new))
property_list_new = list(map(lambda x: x.replace("% crystallinity","crystallinity"), property_list_new))

property_list_new = list(map(lambda x: x.replace("heat deflection temperature: method a: 1.8 mpa (264 psi)","heat deflection temperature"), property_list_new))
property_list_new = list(map(lambda x: x.replace('relative volume loss during a wear test in sand/water-slurry" ; tivar®1000 = 100"','relative volume loss during a wear test in sand/water-slurry'), property_list_new))
property_list_new = list(map(lambda x: x.replace("compression set, 100°c, 22h, type 1, method b","compression set type 1 method b"), property_list_new))
property_list_new = list(map(lambda x: x.replace("dielectric constant, 60hz, 1.93 mm","dielectric constant at 60hz"), property_list_new))
property_list_new = list(map(lambda x: x.replace("dielectric constant, 60hz, 1.96 mm","dielectric constant at 1.96 mm"), property_list_new))
property_list_new = list(map(lambda x: x.replace("compression set, 23°c, 168h, type a","compression set type a"), property_list_new))
property_list_new = list(map(lambda x: x.replace("tear strength, method ba, perpendicular","tear strength"), property_list_new))
property_list_new = list(map(lambda x: x.replace("electric strength, 23°c (ac)","electric strength at 23°c (ac)"), property_list_new))
property_list_new = list(map(lambda x: x.replace("elongation at break elast, perpendicular","perpendicular elongation at break"), property_list_new))
property_list_new = list(map(lambda x: x.replace("electric strength, 23°c (dc)","electric strength at 23°c (dc)"), property_list_new))
property_list_new = list(map(lambda x: x.replace("flame rating, 1.1 mm","flame rating at 1.1 mm"), property_list_new))
property_list_new = list(map(lambda x: x.replace("dissipation factor 23℃、60%rh、1mhz","dissipation factor at 23℃"), property_list_new))



to_remove = ["24 hr",
"charpy notced impact strength",
 "flame rating 3 ",
 "ti density",
 "coefficient of linear thermal expansion (23 to 100°c) (73°f to 210°f)",
 "direction",
 "load",
 "heat deflection temperature 4.6 kgf/cm²",
 "limiting pv at 0.1 / 1 m/s cylindrical sleeve bearings",
 "96% h2so4 (sulphuric acid)",
 "ul94",
 "colour; black (bk)",
 "color no.",
 "flexural modulus 23℃",
 "particles > 250 um",
 "volume resistivity 23°c 50% rh",
 "dissipation factor 23℃、60%rh、1mhz",
 "flexural stress (23°c) iso 178",
 "flammability: ul 94 (3 mm (1/8 in.)) (5)",
 "density 23℃","charpy impact strength notched (double 14°) (14) - -",
 "water absorption 23 °c (69%) rh",
 "impact",
 "tensile strength (9) -",
 "density / specific gravity",
"tensile strain at yield(9) -",
"tensile strength at yield 73°f",
"tensile strain at break (9) -",
"coefficient of linear thermal expansion (-40 to 150 °c) (-40 to 300°f)",
"molding shrinkage flow - (0.126 in (3.20 mm))",
"deflection temperature at 1.8 mpa (264 psi)",
"melt volume-flow rate (mvr) (190°c/2.16 kg)",
"heat deflection temperature a",
"charpy impact strength unnotched (13) -",
"flexural modulus - chord 1",
"izod notched impact strength iso 180/a (-30 °c)",
"mold shrinkage x flow 2mm",
"tensile modulus 80°c",
"vn at 0.5% in sulfuric acid",
"tensile modulus 80°c",
"flexural stress 2",
"electric strength k20/p50",
"clte flow -",
"charpy notched impact strength (23â„ƒ)",
"colour",
"dielectric constant 23℃、60%rh、1mhz",
"arc resistance 2","particles > 300 um",
"flexural strength 23℃","insulation resistance 2 (90°c)","color","gloss",
 "flexural modulus 2","flexural modulus 1","direction)","charpy impact strength notched -",
 "tensile strength 23℃","filler","flexural modulus (160°c)","molding shrinkage 1",
 "ball pressure test 1 ","charpy impact strength double 14° notched -","flammability hb 3mm",
 "tensile strength 80c","compression set, 23°c, 22h, type a","compression set, 100°c, 300h, type a",
     "compression set, 23°c, 22h, type 1, method b","dielectric constant 60hz, 1.96 mm",
 "compression set, 100°c, 70h, type a","flame rating, 1.7 mm","flame rating, 1.0 mm","dielectric strength, 2.0 mm",
"compression set, 23°c, 168h, type 1, method b","dielectric constant, 60hz, 2.03 mm",
"compression set, 100°c, 168h, type 1, method b","volume resistivity, 23°c","dielectric constant, 60hz, 2.01 mm",
"dielectric constant 60hz, 1.98 mm","compression set, 100°c, 168h, type a","volume resistivity, 2.0 mm",
"dielectric constant 60hz","compression set, 100°c, 22h, type a","compression set, 100°c, 70h, type 1, method b",
"compression set, 70°c, 168h, type a","flame rating, 1.6 mm","ompression set, 70°c, 168h, type 1, method b",
"flame rating, 3.0 mm","compression set, 23°c, 70h, type a"
            ]
for x in property_list_new:
    if x in to_remove:
        property_list_new.remove(x)


In [14]:
property_list_new=[x.strip() for x in property_list_new]
property_list_new = list(set(property_list_new))

In [15]:
#MODIFIER
modifiers_list = ['weak', 'lower', 'inferior', 'low', 'very low',
'standard', 'typical', 'good', 'intermediate', 'common', 'fair', 'moderate', 'normal', 'ordinary',
'maximum', 'best', 'superior', 'elevated', 'great', 'exceptional', 'really good', 'outstanding', 'superb', 'high', 'very high', 'very good']

modifiers_list+=list(set(properties['VALUE_ASSMNT_SI'] +" "+properties['UNIT_OF_MEAS_SI']))

In [16]:
#FILLER
FILLER = pd.read_csv("./data/FILLER.csv")
fillers_list = get_unique_values(FILLER, 'Filler Type')
fillers_list = [x.lower() for x in fillers_list]
fillers_list.append("glass fiber load")
fillers_list[0:10]

['glass fiber (short) load',
 'total load',
 'glass bead load',
 'carbon fiber load',
 'glass fiber (long) load',
 'mineral filler load',
 'ptfe filler load',
 'stainless steel fiber load',
 'specialty filler load',
 'aramid fiber load']

In [17]:
#FILLER_PERCENTAGE
FILLER_PERCENTAGE = list(set(FILLER['Filler %']+" %"))

In [18]:
#FEATURE
FEATURE = pd.read_csv("./data/FEATURE.csv")
features_list = get_unique_values(FEATURE, 'FEATURE')
features_list = [x.lower() for x in features_list]
features_list[0:10]

['sustainable',
 'unfilled',
 'heat resistant',
 'thermally conductive',
 'glass reinforced',
 'injection molding',
 'specialty appearance',
 'uv resistant',
 'impact modified',
 'light stabilized']

In [19]:
#COMPETITOR_GRADE
COMPETITOR_DATA = pd.read_csv("./data/COMPETITOR_DATA.csv")
Competitor_Grade_Name = get_unique_values(COMPETITOR_DATA, 'COMPETITOR')
Competitor_Grade_Name = get_unique_values(COMPETITOR_DATA, 'COMPETITOR_GRADE')

In [20]:
#CERTIFICATION
import ast
CERTIFICATION = pd.read_csv("./data/CERTIFICATION.csv")
CERTIFICATION['certifications'] = CERTIFICATION['certifications'].apply(lambda x:ast.literal_eval(x))
certifications_list_combined = CERTIFICATION['certifications'].values.tolist()
certifications_list = sum(certifications_list_combined, [])
certifications_list = list(set(certifications_list))
certifications_list

['GMP.POM.046',
 'CP4240',
 'MS-AR-100 BMV2-HF',
 'TST 055 58.03',
 'MS-AR-100 BM',
 'DBL 5403',
 'Li Auto',
 'CP4624',
 'GMW22P-POM-C2U',
 'Stellantis â€“ Chrysler',
 'PMP 01994_10_00137',
 'EMP 60',
 'TSM5515G-1BL',
 'Q-BJEV 01.33',
 'MS-AR-100 DG',
 'DBL 5562',
 'TL52622 MD03',
 'WSS-M4D840-A6',
 'WSS-M2D378-B1',
 'FRM 18-27-040 /--B',
 'FRM 18-27-134 /---',
 'WSK-M4D840-A1',
 'WSS-M4D865-B5',
 'IVECO 15-5244 EMP70',
 'EMP 90',
 'TST 055 54.32',
 'SMTC 5 310 020',
 'CP4931',
 'GMW16924P-POM-C4',
 'WSF-M4D803-A2',
 'GMW15812, Type 5M',
 'GMW22P-POM-C2',
 '28 B 22-O014',
 'DBL 5404',
 'GMP.POM.041',
 'GMW15812, Type 7M',
 'DBL 5406',
 'TL52622 MD22',
 'CP1986',
 'CP1586',
 'WSK-M4D635-A2',
 'GMW16720P-PE',
 'GMW16278P-POM-Type C3',
 'Mercedes-Benz Group (Daimler)',
 'MS-AR-100 FG',
 'CP5280',
 '28 B 22-X009',
 'FTM69 0012',
 'UB16b',
 'GMW17521P-PPS-GF40-Type 2',
 'TL52277',
 'PPS(B60)-IPL-1',
 'CP4570',
 'TL52622 MD07',
 'PMP 01994_14_00001',
 'FTM69 0008',
 'MS-AR-100 BMV',
 'WSS-M4

In [21]:
applications_list = [x for x in applications_list if not pd.isnull(x)]
applications_list = list(set(applications_list))

Brands = [x for x in Brands if not pd.isnull(x)]
Brands = list(set(Brands))

Polymer = [x for x in Polymer if not pd.isnull(x)]
Polymer = list(set(Polymer))

property_list_new = [x for x in property_list if not pd.isnull(x)]
property_list_new = list(set(property_list_new))

modifiers_list = [x for x in modifiers_list if not pd.isnull(x)]
modifiers_list = list(set(modifiers_list))

FILLER_PERCENTAGE = [x for x in FILLER_PERCENTAGE if not pd.isnull(x)]
FILLER_PERCENTAGE = list(set(FILLER_PERCENTAGE))

features_list = [x for x in features_list if not pd.isnull(x)]
features_list = list(set(features_list))

Competitor_Grade_Name = [x for x in Competitor_Grade_Name if not pd.isnull(x)]
Competitor_Grade_Name = list(set(Competitor_Grade_Name))

certifications_list = [x for x in certifications_list if not pd.isnull(x)]
certifications_list = list(set(certifications_list))

In [22]:
unique_values_dict = {'APPLICATION': applications_list, 'BRAND': Brands, 'POLYMER': Polymer, 'PROPERTY': property_list_new, 'MODIFIER': modifiers_list, 'FILLER': fillers_list, 
          'FILLER_PERCENTAGE': FILLER_PERCENTAGE, 'FEATURE': features_list, 'COMPETITOR_GRADE' : Competitor_Grade_Name, 'CERTIFICATION':certifications_list}

In [23]:
for l, v in unique_values_dict.items():
    v = pd.Series([v])
    if v.isna().sum() > 0:
        print(f"{l}: {v}")

In [24]:
filename = 'unique_values_29_05_23'

In [25]:
json_path = filename + '.json'
with open(json_path, 'w', encoding="utf-8") as fp:
    json.dump(unique_values_dict, fp)

In [26]:
import numpy as np

for v in modifiers_list:
    if isinstance(v, str):
        pass
    elif np.isnan(v):
        print(v)

In [27]:
unique_values_df = pd.DataFrame(dict([(k, pd.Series(v)) for k,v in unique_values_dict.items()]))
unique_values_df

,APPLICATION,BRAND,POLYMER,PROPERTY,MODIFIER,FILLER,FILLER_PERCENTAGE,FEATURE,COMPETITOR_GRADE,CERTIFICATION
0,seal for roof window,CELSTRAN,pom,rti - str @ 1.6mm nom. thickn.,6.4 kJ/m²,glass fiber (short) load,ND %,rotomolding,MILASTOMER S-702B,GMP.POM.046
1,automotive air duct,ECOMID,PA*,"compression set, 100°c, 22h, type 1, method b",500 cm³/g,total load,43.0 %,light stabilized,Pocan A3131,CP4240
2,roller for residential curtain,vectra,PA6,durability,128 MPa,glass bead load,33.0 %,extrusion,Yuhwa Hiden U090,MS-AR-100 BMV2-HF
3,lantern-ring/guide bushing,fortron,pa66,"tensile strain at break, perpendicular",19600 MPa,carbon fiber load,26.0 %,heat resistant,Ultradur S 4090 G6,TST 055 58.03
4,seat adjustment-gear housing,kepital,pp,flow rate,60 % by wt.,glass fiber (long) load,55.0 %,laser markable,Novaduran 5010GT30,MS-AR-100 BM
...,...,...,...,...,...,...,...,...,...,...
11770,button for facial massager,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11771,coil bobbin component for luggage carrier,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11772,reservoir and filter case,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11773,push-pull cable,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
excel_path = filename + '.xlsx'
unique_values_df.to_excel(excel_path)

In [29]:
# !pip install openpyxl